# Emulator intercomparison: deterministic vs CRPS ensemble vs diffusion

One UNet base, one dataset, one set of shared design choices, three ways of turning a state into the next one:

| Emulator | Next state | Trained with |
|---|---|---|
| **deterministic** | one prediction | MSE + spectral loss, pushforward rollouts |
| **CRPS ensemble** | one *sample* per call (noise inputs) | fair CRPS + spectral loss, pushforward rollouts |
| **diffusion** | deterministic mean + *sampled* residual (EDM, as in CorrDiff) | denoising loss, one step |

**Shared by all three** (the comparison varies only the paradigm):
- ACCESS-OM2 1° column heat (OHC) and freshwater content, with surface heat and freshwater fluxes and wind stress as forcing.
- Splits: train 1970–2000, validate 2001–2004, skill test seeded 2005-02 and run to 2018-12.
- Normalisation, and the UNet backbone with residual update, periodic/tripole padding and land filling.
- **The hard global budget constraint:** after every step the global content change equals the surface flux integrated over true month lengths, so no emulator can drift globally.

**Training is split so the shared part is trained once:**

```
shared trunk (deterministic, curriculum 1 -> 4 -> 12 steps)
 ├─ deterministic : + pushforward [24, 12] -> [48, 12]
 ├─ CRPS ensemble : trunk weights + zero-initialised noise inputs, then CRPS with the same pushforward schedule
 └─ diffusion     : the deterministic model (frozen) as the mean + a denoiser trained on its one-step residuals
```

Every trained model is saved under `runs/intercomparison/<name>/weights.pt`. Re-running a training cell loads the saved weights instead of retraining, unless `RETRAIN[name]` is set to `True`.

In [ ]:
# =============================================================================
# 1. Setup
# =============================================================================
import os
import sys
import time

# Let the CUDA allocator grow segments instead of fragmenting memory. Must be set
# before the first CUDA call, so restart the kernel for it to take effect.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

REPO_ROOT = "/home/156/txs156/uom/OM2-emulator"
_starts = [
    Path(p).expanduser()
    for p in (REPO_ROOT, os.environ.get("OM2_EMULATOR_REPO"), globals().get("__vsc_ipynb_file__"), Path.cwd())
    if p
]
_candidates = [q for p in _starts for q in (p, *p.parents)]
SRC = next((q / "src" for q in _candidates if (q / "src" / "Experiment").is_dir()), None)
if SRC is None:
    raise FileNotFoundError(f"Could not find src/Experiment from {[str(p) for p in _starts]}; set REPO_ROOT.")
sys.path.insert(0, str(SRC))

from Experiment import (
    DataConfig, ExperimentConfig, LossConfig, ModelConfig, NormalisationConfig, TimeConfig, TrainConfig,
    WindowConfig, EvalConfig, InitialMonthDataset,
    build_data, build_losses, build_model, build_module, build_trainer, load_trunk, count_parameters,
    build_diffusion, build_diffusion_module,
    run_skill_test, run_control, leading_growth_mode, meridional_transports,
    intercomparison_summary, plot_intercomparison, plot_intercomparison_spectra, plot_intercomparison_transports,
    plot_skill_diagnostics, plot_spectra, plot_meridional_transports, plot_control, plot_rmse_by_epoch,
    plot_growth_mode, persistence_rmse, climatology_rmse,
)

RUNS = SRC.parent / "runs" / "intercomparison"
FIG_DIR = SRC.parent / "figures" / "intercomparison"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def save(fig, name):
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    print(f"Saved {FIG_DIR / name}")


print(f"src: {SRC} | torch {torch.__version__} | device {DEVICE}")

In [ ]:
# =============================================================================
# 2. Configuration: the shared settings, then each branch's changes.
# cfg.with_changes(section={...}) returns a validated copy with those fields changed.
# =============================================================================
shared_cfg = ExperimentConfig(
    data=DataConfig(
        path="/g/data/nm47/txs156/OM2-emulator/data/1deg_ocean_heat_emulator_data.nc",
        # OHC and freshwater content (salt content relative to 35 psu, opposite sign).
        prognostic=["ocean_heat_content_2d", "ocean_freshwater_content_2d"],
        forcing=["total_surface_heat_flx", "ocean_freshwater_flux", "tau_x", "tau_y"],
        normalisation=NormalisationConfig(strategy="Spatial_climatology", fit_period=("1970-01", "2000-12")),
    ),
    time=TimeConfig(
        time_axis=("1970-01", "2018-12"),
        train=("1970-02", "2000-12"),    # windows end by 2000-12
        valid=("2001-01", "2001-12"),    # scored 36 months: targets to 2004-12, before the skill test
        test=("2005-02", "2018-11"),     # skill test: seeded 2005-02, predicts 2005-03 .. 2018-12
        control=("2003-01", "2003-12"),  # stability: this year's forcing repeated
        control_repeats=50,              # 50 years
    ),
    window=WindowConfig(
        n_prior=2,
        posterior_steps=60,
        rollout_steps=12,
        rollout_schedule={0: 1, 5: 4, 15: 12},  # the shared trunk's curriculum
        valid_rollout_steps=36,
    ),
    # Closure terms off: the global budgets are enforced exactly (ClosureConfig.enforce).
    loss=LossConfig(terms={"local_mse": 1.0, "spectral": 2.0}),
    model=ModelConfig(arch="unet", padding_mode="zeros", width_multiplier=4, channels_last=True),
    train=TrainConfig(
        batch_size=16, max_epochs=30, lr=1e-4, lr_scheduler="cosine", precision="16-mixed",
        checkpoint_rollout_steps=True, rollout_steps_in_memory=2, seed=42,
        run_dir=str(RUNS / "shared"),
    ),
    eval=EvalConfig(snapshot_periods=["2005-05", "2010-12", "2015-12"]),
)

# Both rollout branches: pushforward to leads 25-36, then 49-60 months.
PUSHFORWARD = {0: [24, 12], 15: [48, 12]}

configs = {
    "deterministic": shared_cfg.with_changes(
        window={"rollout_schedule": PUSHFORWARD},
        train={"max_epochs": 30, "lr": 5e-5, "run_dir": str(RUNS / "deterministic")},
    ),
    "crps_ensemble": shared_cfg.with_changes(
        paradigm="crps_ensemble",
        loss={"terms": {"crps": 1.0, "spectral": 2.0}},
        ensemble={"noise_channels": 4, "train_members": 4, "eval_members": 10},
        window={"rollout_schedule": PUSHFORWARD},
        # 8 samples x 4 members = 32 rollouts per batch: about 2x the deterministic branch's cost.
        train={"batch_size": 8, "max_epochs": 30, "lr": 5e-5, "run_dir": str(RUNS / "crps_ensemble")},
    ),
    "diffusion": shared_cfg.with_changes(
        paradigm="diffusion",
        diffusion={"sampler_steps": 18, "width_multiplier": 4},
        ensemble={"eval_members": 10},
        window={"rollout_schedule": None},  # one-step training
        train={"max_epochs": 200, "lr": 1e-4, "run_dir": str(RUNS / "diffusion")},
    ),
}

# Set True to retrain a stage even if its saved weights exist.
RETRAIN = {"shared": False, "deterministic": False, "crps_ensemble": False, "diffusion": False}

In [ ]:
# =============================================================================
# 3. Data (one normalisation and cache for every branch)
# =============================================================================
data = build_data(shared_cfg)


def loaders(cfg):
    """Train/valid loaders with cfg's batch size (the branches use different batch sizes)."""
    train = DataLoader(InitialMonthDataset(data.train_indices), batch_size=cfg.train.batch_size, shuffle=True)
    valid = DataLoader(InitialMonthDataset(data.valid_indices), batch_size=cfg.train.batch_size, shuffle=False)
    return train, valid


def weights_path(name):
    return RUNS / name / "weights.pt"


def train_or_load(name, cfg, model, module):
    """Train module (which wraps model) unless saved weights exist; returns (model, training minutes)."""
    path = weights_path(name)
    if path.exists() and not RETRAIN[name]:
        model.load_state_dict(torch.load(path, map_location="cpu"))
        print(f"{name}: loaded {path}")
        return model.to(DEVICE), None
    start = time.perf_counter()
    build_trainer(cfg).fit(module, *loaders(cfg))
    minutes = (time.perf_counter() - start) / 60
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path)
    pd.Series({"training_minutes": minutes}).to_json(path.with_name("timing.json"))
    print(f"{name}: trained in {minutes:.1f} min, saved {path}")
    return model.to(DEVICE), minutes


def free_gpu(*objects):
    for obj in objects:
        del obj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


training_minutes = {}

In [ ]:
# =============================================================================
# 4. Shared trunk: deterministic, curriculum 1 -> 4 -> 12 steps
# =============================================================================
trunk = build_model(shared_cfg, data)
print(f"UNet parameters: {count_parameters(trunk):,}")
module = build_module(shared_cfg, trunk, build_losses(shared_cfg, data), data)
trunk, training_minutes["shared"] = train_or_load("shared", shared_cfg, trunk, module)
trunk_weights = {k: v.detach().cpu().clone() for k, v in trunk.state_dict().items()}
free_gpu(module)

In [ ]:
# =============================================================================
# 5. Branch: deterministic (trunk + pushforward)
# =============================================================================
cfg = configs["deterministic"]
det_model = load_trunk(build_model(cfg, data), trunk_weights)
module = build_module(cfg, det_model, build_losses(cfg, data), data)
det_model, training_minutes["deterministic"] = train_or_load("deterministic", cfg, det_model, module)
free_gpu(module)

In [ ]:
# =============================================================================
# 6. Branch: CRPS ensemble (trunk + zero-initialised noise inputs, CRPS + pushforward)
# =============================================================================
cfg = configs["crps_ensemble"]
ens_model = load_trunk(build_model(cfg, data), trunk_weights)  # starts exactly as the trunk
module = build_module(cfg, ens_model, build_losses(cfg, data), data)
ens_model, training_minutes["crps_ensemble"] = train_or_load("crps_ensemble", cfg, ens_model, module)
free_gpu(module)

In [ ]:
# =============================================================================
# 7. Branch: diffusion (the deterministic branch, frozen, as the mean + a trained denoiser)
# Using the deterministic branch as the mean makes this a clean ablation: diffusion
# = deterministic + a sampled residual. (For the trunk instead, pass `trunk`.)
# =============================================================================
cfg = configs["diffusion"]
mean_model = load_trunk(build_model(shared_cfg, data), det_model.state_dict()).to(DEVICE)
diffusion = build_diffusion(cfg, data, mean_model)
print(f"Denoiser parameters: {count_parameters(diffusion.denoiser):,} | sigma_data = {float(diffusion.sigma_data):.3f}")
module = build_diffusion_module(cfg, diffusion, data)
diffusion, training_minutes["diffusion"] = train_or_load("diffusion", cfg, diffusion, module)
free_gpu(module)

In [ ]:
# =============================================================================
# 8. Skill tests: every emulator from the 2005-02 state, 166 months, real forcing,
# budget constraint on. Stochastic emulators run ensemble.eval_members members.
# =============================================================================
emulators = {"deterministic": det_model, "crps_ensemble": ens_model, "diffusion": diffusion}
skills, inference = {}, {}
for name, model in emulators.items():
    start = time.perf_counter()
    skills[name] = run_skill_test(configs[name], model, data)
    seconds = time.perf_counter() - start
    members = skills[name].attrs["members"]
    years = skills[name].sizes["time"] / 12
    inference[name] = {"members": members, "seconds": seconds, "seconds_per_member_year": seconds / members / years}
    print(f"{name}: {members} member(s) in {seconds:.0f} s")

In [ ]:
# =============================================================================
# 9. Intercomparison: scores and figures
#   rmse/clim : RMSE of the ensemble mean / RMSE of zero anomaly (lower is better; ~1.4 = no skill but realistic)
#   acc       : anomaly correlation of the ensemble mean (higher is better)
#   amplitude_ratio_last_yr : one member's RMS anomaly / the truth's (1 = right variability; < 1 = variance collapse)
#   spread_skill : ensemble spread / ensemble-mean error (1 = reliable; 0 for deterministic)
#   crps      : fair CRPS (MAE for one member; lower is better)
#   global_*  : ocean-integrated anomaly vs truth (RMS error, and trend error as a fraction of the true trend)
# =============================================================================
variables = shared_cfg.data.prognostic
summary = intercomparison_summary(skills, variables)
display(summary.round(3))
summary.to_csv(FIG_DIR / "intercomparison_summary.csv")

save(plot_intercomparison(skills, variables), "intercomparison_skill.png")
save(plot_intercomparison_spectra(skills, variables), "intercomparison_spectra.png")
transports = {name: meridional_transports(configs[name], data, skill) for name, skill in skills.items()}
save(plot_intercomparison_transports(transports), "intercomparison_transports.png")

In [ ]:
# =============================================================================
# 10. Per-emulator diagnostics (one member): skill vs persistence/climatology,
# spectra through the test, implied MHT/MFWT
# =============================================================================
for name, skill in skills.items():
    save(plot_skill_diagnostics(skill, variables), f"{name}_skill_diagnostics.png")
    save(plot_spectra(skill, variables), f"{name}_spectra.png")
    save(plot_meridional_transports(transports[name]), f"{name}_transports.png")
    plt.close("all")

In [ ]:
# =============================================================================
# 11. Stability: 50-year control runs (2003 forcing repeated), one member each,
# and the leading growth mode of the deterministic one-step map
# =============================================================================
controls = {}
for name, model in emulators.items():
    controls[name] = run_control(configs[name], model, data)
    fig = plot_control(controls[name], variables)
    fig.suptitle(f"{name}: control run", fontsize=13)
    save(fig, f"{name}_control.png")
growth = leading_growth_mode(det_model, data, shared_cfg.time.test[0])
print(f"Deterministic leading growth factor: {growth['growth']:.4f} per step")
save(plot_growth_mode(growth, data), "deterministic_growth_mode.png")

In [ ]:
# =============================================================================
# 12. Training curves and cost
# =============================================================================
baselines = {"Validation persistence (z held fixed)": persistence_rmse(shared_cfg, data),
             "Validation climatology (zero anomaly)": climatology_rmse(shared_cfg, data)}
for name in ("shared", "deterministic", "crps_ensemble"):
    run_dir = RUNS / name
    if (run_dir / "metrics.csv").exists():
        fig = plot_rmse_by_epoch(run_dir, variables, baselines=baselines)
        fig.suptitle(name, fontsize=13)
        save(fig, f"{name}_rmse_by_epoch.png")

def minutes(name):
    path = RUNS / name / "timing.json"
    return pd.read_json(path, typ="series")["training_minutes"] if path.exists() else float("nan")

cost = pd.DataFrame({
    name: {
        "trainable parameters": count_parameters(diffusion.denoiser if name == "diffusion" else emulators[name]),
        "training minutes (branch only)": minutes(name),
        "members in skill test": inference[name]["members"],
        "seconds per member-year": inference[name]["seconds_per_member_year"],
    }
    for name in emulators
}).T
cost.loc["shared trunk", "training minutes (branch only)"] = minutes("shared")
display(cost.round(2))
cost.to_csv(FIG_DIR / "intercomparison_cost.csv")